In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
import pickle
import aux_functions as aux
import traceback
importlib.reload(aux)

def read_pkl(path):
    with open(path, "rb") as file:
        return pickle.load(file) 

def write_pkl(var, path):
    with open(path, "wb") as file:
        pickle.dump(var, file)

In [2]:
df_cell = read_pkl("session_data/all_sessions.pkl")
df_cell = df_cell[df_cell['RF_type']=='Focal Foveal']
df_cell

,start,search,mapRF,grid_coord,depth,brain_area,sample,array,prefer,SI,RFin,RFout,coreRFin,,RF_type,session,neuron_id
3,6,m08cat186spk034a,m08det185spk034a,-1+6,1486,TE,0.0,NaN,house,-0.244390,NaN,NaN,NaN,NaN,Focal Foveal,m08cat186,spk034a
4,7,m08cat186spk034b,m08det185spk034b,-1+6,1486,TE,0.0,NaN,house,-0.185990,NaN,NaN,NaN,NaN,Focal Foveal,m08cat186,spk034b
7,12,m08cat186spk036a,m08det185spk036a,-1+6,1496,TE,0.0,NaN,house,-0.259840,NaN,NaN,NaN,NaN,Focal Foveal,m08cat186,spk036a
8,13,m08cat186spk036b,m08det185spk036b,-1+6,1496,TE,0.0,NaN,house,-0.394650,NaN,NaN,NaN,NaN,Focal Foveal,m08cat186,spk036b
14,22,m08cat186spk039b,m08det185spk039b,-1+6,1356,TE,0.0,NaN,face,0.429380,NaN,NaN,NaN,NaN,Focal Foveal,m08cat186,spk039b
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6171,3074,m06cat288spk126z,m06det287spk126z,+2-5,688,V4,0.0,NaN,NaN,0.073817,NaN,NaN,NaN,NaN,Focal Foveal,m06cat288,spk126z
6172,3077,m06cat288spk128a,m06det287spk128a,+2-5,698,V4,0.0,NaN,face,0.041171,NaN,NaN,NaN,NaN,Focal Foveal,m06cat288,spk128a
6173,3078,m06cat288spk128b,m06det287spk128b,+2-5,698,V4,0.0,NaN,NaN,0.103600,NaN,NaN,NaN,NaN,Focal Foveal,m06cat288,spk128b
6174,3079,m06cat288spk128c,m06det287spk128c,+2-5,698,V4,0.0,NaN,NaN,0.046372,NaN,NaN,NaN,NaN,Focal Foveal,m06cat288,spk128c


In [3]:
session_list = list(df_cell['session'].unique())

In [4]:
# start_time = 0
# end_time = 100
# bin_width = 50
# step = 50
# t_ends = np.arange(start_time + bin_width, end_time + step, step)
# print(t_ends)



start_time = -300
end_time = 300
bin_width = 50
step = 10
t_ends = np.arange(start_time + bin_width, end_time + step, step)
print(t_ends)

[-250 -240 -230 -220 -210 -200 -190 -180 -170 -160 -150 -140 -130 -120
 -110 -100  -90  -80  -70  -60  -50  -40  -30  -20  -10    0   10   20
   30   40   50   60   70   80   90  100  110  120  130  140  150  160
  170  180  190  200  210  220  230  240  250  260  270  280  290  300]


In [5]:
list_errors = []
dicio_save = {'V4':{},'IT':{}}
start_timestamp = pd.Timestamp.now()
i=0
SAVE_ITER = 0
for SESSION in session_list:
    i+=1
    now_timestamp = pd.Timestamp.now()
    print(f'session = {SESSION}, i={i}, em tempo {now_timestamp - start_timestamp}')
    print('final previsto: ',(now_timestamp - start_timestamp)*((163-i)/i) + now_timestamp)
    MONKEY = 'M'+SESSION[1:3]
    df_session = df_cell[df_cell['session']==SESSION]
    df_session.index = pd.Index(range(len(df_session.index)))
    # for (areas_used, name_areas_used) in [(['V4'],'V4'),(['TE','TEO'],'IT'),(['V4','TE','TEO'],'V4+IT')]:
    for (areas_used, name_areas_used) in [(['V4'],'V4'),(['TE','TEO'],'IT')]:
        try:
            df_session_areas = df_session[df_session['brain_area'].isin(areas_used)]
            df_session_areas.index = pd.Index(range(len(df_session_areas.index)))
            if len(df_session_areas.index) == 0: continue
            print(f'iniciando decoding {name_areas_used} {SESSION}')
            path_destiny = f'dados/{MONKEY}_neurons/neurons sorted'
            list_neuron_ids = df_session_areas['search']
            list_prefered_cat = df_session_areas['prefer']
            list_brain_area = df_session_areas['brain_area']
            # print(SESSION, areas_used)
            # print(df_session_areas)
            # print(list_prefered_cat)
            df_trials, dicio_firings = aux.get_df_trials_all_saccs_foveal(list_neuron_ids, path_destiny, list_prefered_cat, list_brain_area)
            # print(df_trials)
            ## DECODING DE CATEGORIA DA PROXIMA SACADA
            df_run_model = aux.get_df_run_ml(df_trials, start_time, end_time, bin_width, step, dicio_firings)
            decoding_category = aux.run_temporal_decoding_generic(df_run_model, 'CATEGORY', start_time, bin_width, step, window_size_bins=1)
            # aux.plot_decoding(decoding_category,STR_DETAILS=f'{SESSION} {name_areas_used} Foveal, {len(list_neuron_ids)} Neurons')
            ## DECODING DE CATEGORIA DA PROXIMA SACADA -- SOMENTE FACE E HOUSE
            df_trials2 = df_trials[df_trials['cue_cath'].isin(['face','house'])]
            df_trials2 = df_trials2[df_trials2['sacc_img_cath'].isin(['face','house'])]
            df_run_model2 = aux.get_df_run_ml(df_trials2, start_time, end_time, bin_width, step, dicio_firings)
            decoding_category2 = aux.run_temporal_decoding_generic(df_run_model2, 'CATEGORY', start_time, bin_width, step, window_size_bins=1)
            # aux.plot_decoding(decoding_category2,STR_DETAILS=f'{SESSION} {name_areas_used} Foveal, {len(list_neuron_ids)} Neurons - Somente House e Face')
            dicio_save[name_areas_used][SESSION] = {'df_trials':df_trials,'dicio_firings':dicio_firings, 'df_run_model':df_run_model, 'decoding':decoding_category, 'decoding_restrict':decoding_category2}
        except Exception as e:
            print()
            print('=== ERRO ===')
            print(SESSION, name_areas_used)
            print(traceback.format_exc())
            list_errors.append({'session_id':SESSION, 'areas_used':areas_used, 'name_areas_used':name_areas_used, 'error':e, 'error_complete':traceback.format_exc()})

    if i % 5 == 0:
        print('iniciando save')
        print(pd.Timestamp.now())
        write_pkl([dicio_save, list_errors],f'session_data/foveal_decoding_{SAVE_ITER}.pkl')
        print()
        print('=====')
        print('DADOS SALVOS')
        SAVE_ITER += 1
        list_errors = []
        dicio_save = {'V4':{},'IT':{}}


print('SALVANDO FINAL')
print(pd.Timestamp.now())
write_pkl([dicio_save, list_errors],f'session_data/foveal_decoding_{SAVE_ITER}.pkl')
print('FINALIZADO. DADOS SALVOS.')

session = m08cat186, i=1, em tempo 0 days 00:00:00.000562
final previsto:  2026-10-21 14:56:41.273097
iniciando decoding V4 m08cat186
Iniciando Decoding (5 folds, 56 janelas, 56 bins totais)...
Bins selecionados para Tuning: [25]
--- Processando Fold 1/5 --- 14:57:36
--- Processando Fold 2/5 --- 14:57:44
--- Processando Fold 3/5 --- 14:57:51
--- Processando Fold 4/5 --- 14:57:58
--- Processando Fold 5/5 --- 14:58:06
Iniciando Decoding (5 folds, 56 janelas, 56 bins totais)...
Bins selecionados para Tuning: [25]
--- Processando Fold 1/5 --- 14:58:47
--- Processando Fold 2/5 --- 14:58:51
--- Processando Fold 3/5 --- 14:58:55
--- Processando Fold 4/5 --- 14:58:59
--- Processando Fold 5/5 --- 14:59:03
iniciando decoding IT m08cat186
Iniciando Decoding (5 folds, 56 janelas, 56 bins totais)...
Bins selecionados para Tuning: [25]
--- Processando Fold 1/5 --- 15:00:47
--- Processando Fold 2/5 --- 15:01:03
--- Processando Fold 3/5 --- 15:01:20
--- Processando Fold 4/5 --- 15:01:36
--- Processand

In [ ]:
dicio_save['V4']['m06cat285']['decoding']

{'mean_acc': array([25.56565404, 25.89618555, 25.47904097, 26.28049662, 25.1755228 ,
        26.12596311, 26.46692684, 25.88533674, 26.23991213, 26.5463258 ,
        25.02578682, 27.00764061, 26.96871023, 26.4472223 , 25.50492634,
        25.24230889, 25.59026943, 24.821881  , 25.30294415, 26.14830907,
        25.0000889 , 24.87591183, 25.43944347, 25.64678602, 25.41939276,
        26.40496529, 25.81417675, 25.77308866, 25.82099519, 25.03432329,
        26.20274841, 25.47592777, 25.51548204, 29.12606426, 30.6991826 ,
        30.37272934, 30.62809733, 29.49244811, 29.39860865, 28.71309488,
        28.44389828, 27.15487964, 28.39397363, 26.83761601, 27.70594822,
        27.8034508 , 26.87491808, 26.85961072, 26.66140854, 26.43360394,
        25.61497755, 25.60642055, 25.46313478, 25.89753377, 24.95365137,
        24.92550385]),
 'sem_acc': array([0.31990075, 0.20934793, 0.31610893, 0.11412792, 0.44402297,
        0.52876234, 0.65526229, 0.38497633, 0.40081418, 0.21596328,
        0.39102

In [ ]:
# list_errors, dicio_save
# dicio_save['V4']['m08cat204']
# list_errors

In [ ]:
# dicio_save

In [ ]:
# dicio_save0, list_errors0 = read_pkl(f'session_data/foveal_decoding_0.pkl')
# dicio_save1, list_errors1 = read_pkl(f'session_data/foveal_decoding_1.pkl')
# dicio_save2, list_errors2 = read_pkl(f'session_data/foveal_decoding_2.pkl')

In [ ]:
# # print(dicio_save0['IT'].keys())
# # print(dicio_save1['IT'].keys())
# # print(dicio_save2['IT'].keys())

# list_errors1

[{'session_id': 'm08cat023',
  'areas_used': ['TE', 'TEO'],
  'name_areas_used': 'IT',
  'error': KeyError(nan)},
 {'session_id': 'm08cat030',
  'areas_used': ['TE', 'TEO'],
  'name_areas_used': 'IT',
  'error': NotImplementedError('Please use HDF reader for matlab v7.3 files, e.g. h5py')}]